# CNN vs Transformer Experiment Runner

This notebook is intended for Google Colab. It installs the required packages, runs the full CIFAR-10 experiment pipeline, saves comparison reports and plots, and displays the explainability outputs.

You can choose whether to run the full dataset, the 50% subset, or the 10% subset by selecting a manifest file.

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
requirements_path = PROJECT_ROOT / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(
        "requirements.txt not found. Open the notebook from the repository root in Colab."
    )

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
print("Installed project dependencies from requirements.txt")

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
REPORT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MANIFEST_DIR = PROJECT_ROOT / "manifests"

EPOCHS = 10
BATCH_SIZE = 64
SAMPLE_INDEX = 0
DATA_DIR = "/Users/pritishrv/Documents/Courseworks/NeuralComputing/archive (5)/cifar10"
DATASET_OPTION = "full"  # choose from: full, half, ten_percent

MANIFEST_OPTIONS = {
    "full": MANIFEST_DIR / "cifar10_full.txt",
    "half": MANIFEST_DIR / "cifar10_half.txt",
    "ten_percent": MANIFEST_DIR / "cifar10_ten_percent.txt",
}

if DATASET_OPTION not in MANIFEST_OPTIONS:
    raise ValueError(f"Unknown DATASET_OPTION: {DATASET_OPTION}")

MANIFEST_PATH = MANIFEST_OPTIONS[DATASET_OPTION]
print("Using manifest:", MANIFEST_PATH)

In [ ]:
command = [
    sys.executable,
    "-m",
    "src.run_experiment",
    "--epochs",
    str(EPOCHS),
    "--batch-size",
    str(BATCH_SIZE),
    "--sample-index",
    str(SAMPLE_INDEX),
    "--data-dir",
    DATA_DIR,
    "--manifest-path",
    str(MANIFEST_PATH),
]

completed = subprocess.run(command, check=True, cwd=PROJECT_ROOT)
print("Experiment run finished with return code:", completed.returncode)

In [ ]:
markdown_report_path = REPORT_DIR / "experiment_results.md"
json_report_path = REPORT_DIR / "experiment_results.json"

payload = None
if json_report_path.exists():
    payload = json.loads(json_report_path.read_text())

if markdown_report_path.exists():
    display(Markdown(markdown_report_path.read_text()))
else:
    print("Markdown report not found:", markdown_report_path)

if payload is not None:
    print(json.dumps(payload, indent=2))
else:
    print("JSON report not found:", json_report_path)

In [ ]:
plot_paths = [] if payload is None else payload.get("plots", [])
for plot_path in plot_paths:
    path = Path(plot_path)
    if path.exists():
        display(Markdown(f"## Plot: {path.name}"))
        display(Image(filename=str(path)))
    else:
        print("Plot not found:", path)

In [ ]:
for model_name in ["cnn", "vit"]:
    image_path = OUTPUT_DIR / f"{model_name}_sample_{SAMPLE_INDEX}.png"
    if image_path.exists():
        display(Markdown(f"## {model_name.upper()} Explainability Output"))
        display(Image(filename=str(image_path)))
    else:
        print("Explainability image not found:", image_path)